In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from prophet import Prophet
from sklearn.metrics import mean_absolute_error

In [7]:
# 1. Baca data dari file CSV
df = pd.read_csv("BrentOilPrices.csv")

# 2. Bersihkan nama kolom dari spasi yang tidak terlihat
df.columns = df.columns.str.strip()

# 3. Tentukan nama kolom secara langsung (Manual Fix)
# Kita beri tahu Python bahwa kolom tanggal bernama 'Date' dan harga bernama 'Price'
col_date = "Date" 
col_price = "Price"

# 4. Ubah format kolom tanggal agar bisa dibaca AI
df[col_date] = pd.to_datetime(df[col_date])

# 5. Siapkan tabel baru (df_prophet) dengan nama kolom standar Prophet: 'ds' dan 'y'
df_prophet = df.rename(columns={col_date: 'ds', col_price: 'y'})

# 6. Buang data yang kosong dan urutkan berdasarkan tanggal
df_prophet = df_prophet.sort_values('ds').dropna()

# 7. Tampilkan 5 data teratas untuk memastikan tidak ada error lagi
df_prophet.head()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_11240\1378488101.py:13: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



,ds,y
0,1987-05-20,18.63
1,1987-05-21,18.45
2,1987-05-22,18.55
3,1987-05-25,18.60
4,1987-05-26,18.63


In [12]:
# 1. Inisialisasi model Prophet
model = Prophet()

# 2. Melatih model dengan data df_prophet yang tadi sudah muncul di Step 2
# Pastikan sel ini dijalankan sampai muncul pesan 'Optimization terminated successfully'
model.fit(df_prophet)

# 3. Cetak konfirmasi
print("Model berhasil dilatih dan siap melakukan prediksi!")

14:33:23 - cmdstanpy - INFO - Chain [1] start processing
14:33:26 - cmdstanpy - INFO - Chain [1] done processing


Model berhasil dilatih dan siap melakukan prediksi!


In [16]:
# 1. Melakukan prediksi menggunakan model yang sudah dilatih
# Proses ini akan menghitung nilai prediksi (yhat), batas bawah (yhat_lower), dan batas atas (yhat_upper)
forecast_2032 = model.predict(future_2032)

# 2. Membersihkan urutan data (PENTING: Agar grafik tidak berantakan/garis vertikal)
forecast_2032 = forecast_2032.sort_values('ds').reset_index(drop=True)

# 3. Menampilkan kolom penting dari hasil prediksi
# ds = Tanggal, yhat = Prediksi Harga, yhat_lower/upper = Rentang Ketidakpastian
forecast_2032[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

,ds,yhat,yhat_lower,yhat_upper
11926,2030-11-08,127.583108,-6.256409,251.665632
11927,2030-11-09,129.264492,-1.836033,255.412280
11928,2030-11-10,129.225075,0.159503,253.750032
11929,2030-11-11,127.473970,0.263338,251.594476
11930,2030-11-12,127.328219,-6.305657,252.596708


In [17]:
import plotly.graph_objects as go

# 1. Membuat kanvas grafik
fig = go.Figure()

# 2. Menambahkan area "Pita Pink" (Rentang Ketidakpastian)
# yhat_upper adalah batas atas tebakan AI
fig.add_trace(go.Scatter(
    x=forecast_2032['ds'], y=forecast_2032['yhat_upper'],
    line=dict(width=0), showlegend=False, hoverinfo="skip"
))

# yhat_lower adalah batas bawah tebakan AI
fig.add_trace(go.Scatter(
    x=forecast_2032['ds'], y=forecast_2032['yhat_lower'],
    line=dict(width=0), fill='tonexty', 
    fillcolor='rgba(255, 0, 0, 0.15)', name='Rentang Ketidakpastian'
))

# 3. Menambahkan garis merah (Hasil Prediksi Utama)
fig.add_trace(go.Scatter(
    x=forecast_2032['ds'], y=forecast_2032['yhat'],
    mode='lines', name='Prediksi AI (Prophet)', line=dict(color='red', width=2)
))

# 4. Menambahkan titik-titik hitam (Data Asli/Aktual)
fig.add_trace(go.Scatter(
    x=df_prophet['ds'], y=df_prophet['y'],
    mode='markers', name='Data Aktual', 
    marker=dict(color='black', size=1.5, opacity=0.3)
))

# 5. Mengatur tampilan (Judul dan Nama Sumbu)
fig.update_layout(
    title='Proyeksi Harga Minyak Brent hingga Tahun 2032',
    xaxis_title='Tahun',
    yaxis_title='USD/Barrel',
    hovermode='x unified',
    template='plotly_white'
)

# Tampilkan Grafik
fig.show()